### Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import os
import pandas as pd
import re
import time
from datetime import datetime

import src.main as main

In [ ]:
exec_datetime = datetime.now().strftime("%Y%m%d%H%M%S")
print(exec_datetime)

In [3]:
headers = {
    "User-Agent": "EsportUsernames/1.0 (https://github.com/hollowscene)"
}

In [ ]:
with open("main_wikis.txt", "r") as file:
    main_wikis = [line.strip() for line in file]

## Ingest

In [ ]:
def ingest(wikis: list, pattern: str = re.compile(r".+\ \(.+\)$"), save_as_csv: bool = False, csv_path: str = "data", csv_name: str = None):
    """docstring."""
    df = None
    exec_datetime = datetime.now().strftime("%Y%m%d%H%M%S")

    for wiki in wikis:
        players = main.get_player_pages(wiki=wiki, headers=headers)
        df_wiki = pd.DataFrame.from_dict(players)

        # Player names only have a namespace value of 0
        df_wiki = df_wiki[df_wiki["ns"] == 0]

        # Create new columns
        df_wiki["cleantitle"] = df_wiki.apply(lambda row: main.clean_username(pattern, row["title"]), axis=1)
        df_wiki["cleantitleupper"] = df_wiki.apply(lambda row: main.clean_username(pattern, row["title"]).upper(), axis=1)
        df_wiki["wiki"] = wiki

        if df is None:
            df = df_wiki
        else:
            df.append(df_wiki, ignore_index=True)

    if save_as_csv:
        if csv_name is None:
            exec_datetime = datetime.now().strftime("%Y%m%d%H%M%S")
            csv_name = f"players_{exec_datetime}.csv"
        df.to_csv(os.path.join(csv_path, csv_name), index=False)

    return df

In [ ]:
ingest(["dota2"], save_as_csv=True)

In [ ]:
for wiki in main_wiki_list:
    start_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"{start_datetime} - {wiki} import and clean start")
    ingest_and_clean(wiki, data_subfolder=data_subfolder)
    end_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"{end_datetime} - {wiki} import and clean finished")
    time.sleep(0.5)

In [ ]:
    # print(f"Total player name count in {wiki} wiki: {len(player_names):,}")
    # print(f"Unique player name count in {wiki} wiki: {len(value_counts):,}")

## Analysis

In [ ]:
# Re-use cached imported data
exec_datetime = "20241206223049"
data_subfolder = os.path.join("data", exec_datetime)

In [ ]:
player_names_by_wiki = {}

In [ ]:
for wiki in main_wiki_list:
    # with open(os.path.join(data_subfolder, f"{wiki}_player_names_cleaned.txt")) as f_cleaned:
    #     cleaned_player_names = json.load(f_cleaned)
    # value_counts = Counter([player_name for player_name in cleaned_player_names])
    # player_names_by_wiki[wiki] = value_counts

    # Transform so player name is key
    with open(os.path.join(data_subfolder, f"{wiki}_player_names_cleaned.txt")) as f_cleaned:

        cleaned_player_names = json.load(f_cleaned)

        for player_name in cleaned_player_names:

            if player_name not in player_names_by_wiki:
                player_names_by_wiki[player_name] = {}
                player_names_by_wiki[player_name]["total"] = 0

            if wiki not in player_names_by_wiki[player_name]:
                player_names_by_wiki[player_name][wiki] = 0

            player_names_by_wiki[player_name]["total"] += 1
            player_names_by_wiki[player_name][wiki] += 1

In [ ]:
player_names_by_wiki

In [ ]:
len(player_names_by_wiki)

In [ ]:
player_names_by_wiki["Rich"]

Note: Rich has a page on both the League of Legends and Heroes of the Storm wikis. Consolidation may require parsing of the page content.